# C03 — Embeddings e Busca Semântica

Geração de embeddings e busca por similaridade vetorial com FAISS e sentence-transformers.

In [ ]:
%pip install sentence-transformers faiss-cpu numpy pandas -q

## 1. Geração de embeddings com Sentence-Transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

frases = [
    "O gato está dormindo no sofá.",
    "Um felino descansa na poltrona.",
    "O cachorro correu no parque.",
    "Inteligência artificial transforma indústrias.",
    "Machine learning é uma subárea da IA.",
]

embeddings = model.encode(frases, normalize_embeddings=True)
print(f"Shape dos embeddings: {embeddings.shape}")

## 2. Similaridade por cosseno manual

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))

query = "Um animal descansando em casa."
query_emb = model.encode([query], normalize_embeddings=True)[0]

scores = [(frases[i], cosine_similarity(query_emb, embeddings[i])) for i in range(len(frases))]
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Query: {query}\n")
for frase, score in scores:
    print(f"  {score:.4f}  {frase}")

## 3. Índice FAISS para busca eficiente

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product = cosseno para vetores normalizados
index.add(embeddings.astype(np.float32))

print(f"Documentos indexados: {index.ntotal}")

In [ ]:
k = 3
distances, indices = index.search(query_emb.reshape(1, -1).astype(np.float32), k)

print(f"Top-{k} resultados para: '{query}'\n")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"  #{rank} [{dist:.4f}] {frases[idx]}")

## 4. Visualização com PCA

In [ ]:
%pip install matplotlib scikit-learn -q

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 5))
plt.scatter(coords[:, 0], coords[:, 1], s=100)
for i, frase in enumerate(frases):
    plt.annotate(frase[:30], (coords[i, 0], coords[i, 1]), fontsize=8, ha='right')
plt.title("Embeddings projetados em 2D (PCA)")
plt.tight_layout()
plt.show()